In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_parquet('/content/drive/MyDrive/client_id.parquet')


In [3]:
df = df.dropna(subset=['target'])

In [4]:
df.shape

(8914963, 42)

In [5]:
df.head()

,date,time,client_id_x,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed
0,2010-01-01,00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,...,Mastercard,Debit (Prepaid),5497590243197280,2022-07,306,True,2,55.0,2008-05,2008
1,2010-01-01,00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,...,Mastercard,Credit,5175842699412235,2024-12,438,True,1,9100.0,2005-09,2015
2,2010-01-01,00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084.0,...,Mastercard,Debit,5874992802287595,2020-05,256,True,1,14802.0,2006-01,2008
4,2010-01-01,00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776.0,...,Visa,Debit,4354185735186651,2020-01,120,True,1,19113.0,2009-07,2014
5,2010-01-01,00:07:00,1807,165,4.81,Swipe Transaction,20519,Bronx,NY,10464.0,...,Mastercard,Debit (Prepaid),5207231566469664,2014-03,198,True,1,89.0,2008-01,2015


In [6]:
df.columns

Index(['date', 'time', 'client_id_x', 'card_id', 'amount', 'use_chip',
       'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc',
       'errors', 'abs_amount', 'target', 'target_numeric', 'amount_bins',
       'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender',
       'address', 'latitude', 'longitude', 'per_capita_income',
       'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards',
       'age_group', 'income_group', 'debt_group', 'card_brand', 'card_type',
       'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued',
       'credit_limit', 'acct_open_date', 'year_pin_last_changed'],
      dtype='object')

In [7]:
df['date'].dtype

dtype('<M8[ns]')

In [8]:
df.drop(columns=[
    'card_id',
    'merchant_id',
    'card_number',
    'address',
    'target',
    'amount_bins',
    'birth_year',
    'birth_month',
    'address',
    'age_group',
    'income_group',
    'debt_group',
    'card_number',
    'expires',
    'cvv'
], inplace=True)

In [9]:
df.drop(columns=['zip', 'retirement_age'], inplace=True)

In [10]:
df.columns

Index(['date', 'time', 'client_id_x', 'amount', 'use_chip', 'merchant_city',
       'merchant_state', 'mcc', 'errors', 'abs_amount', 'target_numeric',
       'current_age', 'gender', 'latitude', 'longitude', 'per_capita_income',
       'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards',
       'card_brand', 'card_type', 'has_chip', 'num_cards_issued',
       'credit_limit', 'acct_open_date', 'year_pin_last_changed'],
      dtype='object')

In [11]:
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce')

In [12]:
df.to_parquet('/content/drive/MyDrive/Machine Learning/Projects/Fraud Detection Capstone Project/Datasets/Cleaned Dataset/df_8.9M.parquet')

In [13]:
df['debt_to_income'] = df['total_debt']/df['yearly_income']
df['available_credit_per_card'] = df['credit_limit']/df['num_credit_cards']

df['hour'] = df['time'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['weekend_flag'] = (df['day_of_week'] >= 5).astype(int)

In [14]:
df[['amount','abs_amount']].corr()

,amount,abs_amount
amount,1.000000,0.724734
abs_amount,0.724734,1.000000


In [15]:
df['is_refund'] = (df['amount'] < 0).astype(int)
df.drop('amount', inplace=True, axis=1)

In [16]:
import datetime

current_year = datetime.datetime.now().year

df['acct_open_date'] = pd.to_datetime(df['acct_open_date'])
df['account_age_years'] = df['date'].dt.year - df['acct_open_date'].dt.year
df['years_since_pin_change'] = df['date'].dt.year - df['year_pin_last_changed']


df['credit_limit_to_income'] = df['credit_limit'] / df['yearly_income']



df.drop(columns=['latitude', 'longitude', 'acct_open_date', 'year_pin_last_changed'], inplace=True)


In [17]:
df.columns

Index(['date', 'time', 'client_id_x', 'use_chip', 'merchant_city',
       'merchant_state', 'mcc', 'errors', 'abs_amount', 'target_numeric',
       'current_age', 'gender', 'per_capita_income', 'yearly_income',
       'total_debt', 'credit_score', 'num_credit_cards', 'card_brand',
       'card_type', 'has_chip', 'num_cards_issued', 'credit_limit',
       'debt_to_income', 'available_credit_per_card', 'hour', 'day_of_week',
       'weekend_flag', 'is_refund', 'account_age_years',
       'years_since_pin_change', 'credit_limit_to_income'],
      dtype='object')

In [18]:
client_stats = df.groupby('client_id_x')['abs_amount'].agg(
    customer_avg_amount='mean',
    customer_std_amount='std',
    customer_txn_count='count'
)

df = df.merge(client_stats, on='client_id_x', how='left')

In [19]:
df['amount_vs_avg'] = df['abs_amount'] / df['customer_avg_amount']

df['amount_zscore'] = (
    (df['abs_amount'] - df['customer_avg_amount'])
    / df['customer_std_amount']
)

68% data → within 1 std  
95% data → within 2 std  
99.7% data → within 3 std  

z_score > 3 often treated as an anomaly.

In [20]:
df.head(2)

,date,time,client_id_x,use_chip,merchant_city,merchant_state,mcc,errors,abs_amount,target_numeric,...,weekend_flag,is_refund,account_age_years,years_since_pin_change,credit_limit_to_income,customer_avg_amount,customer_std_amount,customer_txn_count,amount_vs_avg,amount_zscore
0,2010-01-01,1900-01-01 00:01:00,1556,Swipe Transaction,Beulah,ND,5499,None,77.00,0.0,...,0,1,2,2,0.001139,68.612854,77.071114,3474,1.122239,0.108823
1,2010-01-01,1900-01-01 00:02:00,561,Swipe Transaction,Bettendorf,IA,5311,None,14.57,0.0,...,0,0,5,-5,0.246927,36.653263,60.056309,7842,0.397509,-0.367709


In [21]:
time_strings = df['time'].dt.strftime('%H:%M:%S')

df['transaction_datetime'] = pd.to_datetime(
    df['date'].astype(str) + ' ' + time_strings,
    format='%Y-%m-%d %H:%M:%S'
)

In [22]:
df['time'] = df['time'].dt.time

In [23]:
import gc

vars_to_clear = ['client_stats', 'time_strings']

for var in vars_to_clear:
    if var in locals():
        del locals()[var]

gc.collect()

print("Garbage collection completed. RAM should be cleaner now.")

Garbage collection completed. RAM should be cleaner now.


In [24]:
df = df.sort_values(['client_id_x', 'transaction_datetime'])

In [25]:
df.head()

,date,time,client_id_x,use_chip,merchant_city,merchant_state,mcc,errors,abs_amount,target_numeric,...,is_refund,account_age_years,years_since_pin_change,credit_limit_to_income,customer_avg_amount,customer_std_amount,customer_txn_count,amount_vs_avg,amount_zscore,transaction_datetime
1198,2010-01-01,13:10:00,0,Swipe Transaction,Portland,ME,5942,None,33.959999,0.0,...,0,3,-4,0.295238,61.847805,79.603157,8646,0.549090,-0.350335,2010-01-01 13:10:00
2001,2010-01-01,19:39:00,0,Swipe Transaction,Scarborough,ME,5812,None,7.780000,0.0,...,0,3,-4,0.295238,61.847805,79.603157,8646,0.125793,-0.679217,2010-01-01 19:39:00
2219,2010-01-01,22:13:00,0,Swipe Transaction,Scarborough,ME,5813,None,65.860001,0.0,...,0,3,-4,0.295238,61.847805,79.603157,8646,1.064872,0.050402,2010-01-01 22:13:00
5855,2010-01-03,15:44:00,0,Swipe Transaction,Scarborough,ME,5499,None,1.370000,0.0,...,0,3,-4,0.295238,61.847805,79.603157,8646,0.022151,-0.759741,2010-01-03 15:44:00
7055,2010-01-04,08:49:00,0,Online Transaction,Online,None,4814,None,167.619995,0.0,...,0,3,-4,0.295238,61.847805,79.603157,8646,2.710201,1.328744,2010-01-04 08:49:00


In [ ]:
df['txn_counts_last_24h'] = df.groupby('client_id_x').rolling('24h', on='transaction_datetime')['abs_amount'].count().reset_index(level=0, drop=True).astype('int32')-1